In [0]:
dbutils.widgets.dropdown(name='environment',defaultValue='dev',choices=['dev','prod','qa'],label='select Environment')
env=dbutils.widgets.get('environment')
print(env)

In [0]:
 goldTable=f"saleslake_{env}.gold_{env}.tgtmonthly"
 print(goldTable)
 Silvertable=f"salesLake_{env}.silver_{env}.CleanseMonthlysale"
 print(Silvertable)

In [0]:
spark.sql(f"""
merge into {goldTable} as tgt
using (select * from {Silvertable}
where ingest_ts>
              (select coalesce(max(last_updt_ts),to_timestamp
              ('1900-01-01','yyyy-MM-dd'))
              from {goldTable}
              
              ))

as src
on tgt.sale_id = src.sale_id
when matched and(
tgt.product <> src.product or
tgt.category <> src.category or
tgt.quantity <> src.quantity or
tgt.price <> src.price or
tgt.sale_date <> src.sale_date or
tgt.region <> src.region
)
then update set
 tgt.product = src.product,
tgt.category = src.category,
tgt.quantity = src.quantity,
tgt.price = src.price,
tgt.sale_date = src.sale_date,
tgt.region = src.region,
tgt.last_updt_ts = current_timestamp()
when not matched then insert(
    sale_id,
    product,
    category,
    quantity,
    price,
    sale_date,
    region,
    initial_load_ts,
    last_updt_ts
)

values(
src.sale_id,
src.product,
src.category,
src.quantity,
src.price,
src.sale_date,
src.region,
current_timestamp(),
current_timestamp())
""")



